# Gradient Field CNN — Fine-Tuning

Two-phase fine-tuning of the v3 checkpoint:
1. **Head Tune** — freeze backbone, train classifier head (lr=1e-4, 5 epochs)
2. **Full Fine-Tune** — unfreeze all, train end-to-end (lr=3e-5, 20 epochs)

Dataset: AI-GenBench benchmark_ten_percent subset

---
## 1. Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_DATA_DIR = "/content/drive/MyDrive/datasets"

In [ ]:
import os
import time
import math
import json
import random
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score, roc_curve, precision_recall_curve,
    average_precision_score, classification_report
)
import numpy as np
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

if not torch.cuda.is_available():
    raise RuntimeError(
        '❌ CUDA not available!\n'
        'Go to Runtime → Change runtime type → Select GPU (T4 recommended)'
    )
print(f'✅ GPU available: {torch.cuda.get_device_name(0)}')
print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device = "cuda"
ARTIFACTS_DIR = Path("models")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# DataLoader config
NUM_WORKERS = 2
PIN_MEMORY = True

---
## 2. Fine-Tune Config

In [ ]:
@dataclass
class FinetuneConfig:
    # Model (must match checkpoint)
    depth: int = 4
    base_filters: int = 32
    embedding_dim: int = 128

    # Phase 1: Head Tune (backbone frozen)
    head_lr: float = 1e-4
    head_epochs: int = 5

    # Phase 2: Full Fine-Tune (all unfrozen)
    full_lr: float = 3e-5
    full_epochs: int = 20
    warmup_epochs: int = 2
    min_lr: float = 1e-7

    # Shared
    weight_decay: float = 1e-3
    dropout: float = 0.15
    batch_size: int = 64
    patience: int = 5
    mixup_alpha: float = 0.2
    grad_clip: float = 1.0
    checkpoint_name: str = "gradient_field_cnn_v3_finetuned.pth"

cfg = FinetuneConfig()
print("Fine-tune config:")
for k, v in asdict(cfg).items():
    print(f"  {k}: {v}")

---
## 3. Model & Metrics Definitions

In [ ]:
@dataclass
class Metrics:
    """Container for all classification metrics."""
    accuracy: float = 0.0
    precision: float = 0.0
    recall: float = 0.0
    f1: float = 0.0
    specificity: float = 0.0
    auroc: float = 0.0
    avg_precision: float = 0.0

    labels: np.ndarray = field(default_factory=lambda: np.array([]))
    preds: np.ndarray = field(default_factory=lambda: np.array([]))
    probs: np.ndarray = field(default_factory=lambda: np.array([]))

    def __repr__(self):
        return (
            f"Metrics(acc={self.accuracy:.4f}, prec={self.precision:.4f}, "
            f"rec={self.recall:.4f}, f1={self.f1:.4f}, spec={self.specificity:.4f}, "
            f"auroc={self.auroc:.4f}, ap={self.avg_precision:.4f})"
        )

    def to_dict(self):
        return {
            'accuracy': self.accuracy, 'precision': self.precision,
            'recall': self.recall, 'f1': self.f1, 'specificity': self.specificity,
            'auroc': self.auroc, 'avg_precision': self.avg_precision
        }


def compute_metrics(labels, preds, probs):
    metrics = Metrics(labels=labels, preds=preds, probs=probs)
    if len(labels) == 0:
        return metrics
    metrics.accuracy = accuracy_score(labels, preds)
    metrics.precision = precision_score(labels, preds, zero_division=0)
    metrics.recall = recall_score(labels, preds, zero_division=0)
    metrics.f1 = f1_score(labels, preds, zero_division=0)
    cm = confusion_matrix(labels, preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        metrics.specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    if len(np.unique(labels)) > 1:
        try:
            metrics.auroc = roc_auc_score(labels, probs)
            metrics.avg_precision = average_precision_score(labels, probs)
        except ValueError:
            metrics.auroc = float('nan')
            metrics.avg_precision = float('nan')
    else:
        metrics.auroc = float('nan')
        metrics.avg_precision = float('nan')
    return metrics

In [ ]:
class CompactGradientNet(nn.Module):
    def __init__(self, depth=4, base_filters=32, dropout=0.1, embedding_dim=128):
        super().__init__()
        self.N_GRAD_CHANNELS = 6

        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                               dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                               dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

        gaussian = torch.tensor([[1, 4, 6, 4, 1], [4, 16, 24, 16, 4],
                                  [6, 24, 36, 24, 6], [4, 16, 24, 16, 4],
                                  [1, 4, 6, 4, 1]], dtype=torch.float32) / 256.0
        self.register_buffer('gaussian', gaussian.view(1, 1, 5, 5))

        self.input_norm = nn.BatchNorm2d(self.N_GRAD_CHANNELS)
        self.channel_mix = nn.Sequential(
            nn.Conv2d(self.N_GRAD_CHANNELS, self.N_GRAD_CHANNELS, kernel_size=1),
            nn.ReLU(),
        )

        layers = []
        in_ch = self.N_GRAD_CHANNELS
        for i in range(depth):
            out_ch = base_filters * (2**i)
            layers.extend([
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2)
            ])
            if dropout > 0:
                layers.append(nn.Dropout2d(dropout))
            in_ch = out_ch

        self.cnn = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.embedding = nn.Linear(out_ch, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, 1)

    def compute_gradient_field(self, luminance):
        G_x = F.conv2d(luminance, self.sobel_x, padding=1)
        G_y = F.conv2d(luminance, self.sobel_y, padding=1)
        magnitude = torch.sqrt(G_x**2 + G_y**2 + 1e-8)
        angle = torch.atan2(G_y, G_x)
        sin_angle = torch.sin(angle)
        cos_angle = torch.cos(angle)
        Gxx, Gxy, Gyy = G_x * G_x, G_x * G_y, G_y * G_y
        Sxx = F.conv2d(Gxx, self.gaussian, padding=2)
        Sxy = F.conv2d(Gxy, self.gaussian, padding=2)
        Syy = F.conv2d(Gyy, self.gaussian, padding=2)
        trace = Sxx + Syy
        det_term = torch.sqrt((Sxx - Syy)**2 + 4 * Sxy**2 + 1e-8)
        lambda1, lambda2 = 0.5 * (trace + det_term), 0.5 * (trace - det_term)
        coherence = ((lambda1 - lambda2) / (lambda1 + lambda2 + 1e-8))**2
        magnitude_scaled = torch.log1p(magnitude * 10)
        return torch.cat([G_x, G_y, magnitude_scaled, sin_angle, cos_angle, coherence], dim=1)

    def forward(self, luminance):
        x = self.compute_gradient_field(luminance)
        x = self.input_norm(x)
        x = self.channel_mix(x)
        x = self.cnn(x)
        x = self.global_pool(x).flatten(1)
        emb = self.embedding(x)
        logit = self.classifier(emb)
        return logit.squeeze(1), emb

---
## 4. Dataset & Augmentation

In [ ]:
class LuminanceDataset(Dataset):
    """Base dataset: loads images and converts to luminance."""
    def __init__(self, img_paths, labels, img_size=224):
        self.img_paths = img_paths
        self.labels = labels
        self.img_size = img_size
        self.resize = transforms.Resize((img_size, img_size))
        self.R_COEFF, self.G_COEFF, self.B_COEFF = 0.2126, 0.7152, 0.0722

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        img = self.resize(img)
        img_tensor = TF.to_tensor(img)
        luminance = (self.R_COEFF * img_tensor[0] +
                     self.G_COEFF * img_tensor[1] +
                     self.B_COEFF * img_tensor[2]).unsqueeze(0)
        return luminance.float(), torch.tensor(self.labels[idx], dtype=torch.float32)


class AugmentedLuminanceDataset(LuminanceDataset):
    """Training dataset with augmentations applied before luminance conversion."""
    def __init__(self, img_paths, labels, img_size=224):
        super().__init__(img_paths, labels, img_size)
        self.augment = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
        ])

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        img = self.resize(img)
        img = self.augment(img)
        # JPEG compression simulation
        if random.random() < 0.3:
            from io import BytesIO
            buffer = BytesIO()
            quality = random.randint(70, 95)
            img.save(buffer, format='JPEG', quality=quality)
            buffer.seek(0)
            img = Image.open(buffer).convert('RGB')
        img_tensor = TF.to_tensor(img)
        luminance = (self.R_COEFF * img_tensor[0] +
                     self.G_COEFF * img_tensor[1] +
                     self.B_COEFF * img_tensor[2]).unsqueeze(0)
        return luminance.float(), torch.tensor(self.labels[idx], dtype=torch.float32)


def mixup_batch(x, y, alpha=0.2):
    """Apply Mixup to a batch."""
    if alpha <= 0:
        return x, y
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # ensure lam >= 0.5
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    y_mix = lam * y + (1 - lam) * y[idx]
    return x_mix, y_mix

---
## 5. Benchmark Dataset Loading

Uses `benchmark_ten_percent` from Google Drive.
Structure: `{train,test,validation}/{real,fake}/{raise,laion,coco}/`

In [ ]:
BENCHMARK_DIR = Path(GDRIVE_DATA_DIR) / "benchmark_ten_percent"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}


def get_image_paths_and_labels(split_folder):
    """
    Collect image paths and labels from a benchmark split folder.

    Structure: split_folder/{real,fake}/{raise,laion,coco}/*.jpg
    Label 0 = real, 1 = fake.
    """
    split_folder = Path(split_folder)
    if not split_folder.exists():
        print(f"⚠️  Split folder not found: {split_folder}")
        return [], []
    paths, labels = [], []
    for label, cls in enumerate(["real", "fake"]):
        cls_dir = split_folder / cls
        if not cls_dir.exists():
            print(f"  ⚠️  Missing class dir: {cls_dir}")
            continue
        # Walk into source subdirs (raise, laion, coco, etc.)
        for source_dir in sorted(cls_dir.iterdir()):
            if source_dir.is_dir():
                files = [f for f in source_dir.iterdir()
                         if f.is_file() and f.suffix.lower() in IMG_EXTS]
                for f in files:
                    paths.append(str(f))
                    labels.append(label)
                print(f"  {cls}/{source_dir.name}: {len(files)} images")
            elif source_dir.is_file() and source_dir.suffix.lower() in IMG_EXTS:
                # Handle images directly in real/fake (no source subdir)
                paths.append(str(source_dir))
                labels.append(label)
    return paths, labels


print(f"📂 Loading benchmark dataset from: {BENCHMARK_DIR}\n")

print("--- Train split ---")
train_paths, train_labels = get_image_paths_and_labels(BENCHMARK_DIR / "train")

print("\n--- Validation split ---")
val_paths, val_labels = get_image_paths_and_labels(BENCHMARK_DIR / "validation")

print("\n--- Test split ---")
test_paths, test_labels = get_image_paths_and_labels(BENCHMARK_DIR / "test")

# Augmented training set, standard val/test sets
train_dataset = AugmentedLuminanceDataset(train_paths, train_labels)
val_dataset   = LuminanceDataset(val_paths, val_labels)
test_dataset  = LuminanceDataset(test_paths, test_labels)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f"\n{'='*50}")
print(f"Train: {len(train_dataset)} samples ({sum(train_labels)} fake, {len(train_labels)-sum(train_labels)} real)")
print(f"Val:   {len(val_dataset)} samples ({sum(val_labels)} fake, {len(val_labels)-sum(val_labels)} real)")
print(f"Test:  {len(test_dataset)} samples ({sum(test_labels)} fake, {len(test_labels)-sum(test_labels)} real)")

---
## 6. Training & Evaluation Helpers

In [ ]:
def one_epoch(model, loader, optimizer, criterion, device, use_mixup=False, mixup_alpha=0.2, grad_clip=0.0):
    """Train for one epoch. Returns (mean_loss, train_accuracy)."""
    model.train()
    losses, correct, total = [], 0, 0
    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if use_mixup:
            x, y = mixup_batch(x, y, alpha=mixup_alpha)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits, y)
        loss.backward()
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        losses.append(loss.item())
        preds = (torch.sigmoid(logits) >= 0.5).float()
        if not use_mixup:
            correct += (preds == y).sum().item()
            total += y.size(0)
        else:
            correct += (preds == (y >= 0.5).float()).sum().item()
            total += y.size(0)
    return np.mean(losses), correct / total if total > 0 else 0.0


@torch.inference_mode()
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    all_labels, all_probs = [], []
    for x, y in tqdm(loader, desc="Evaluating", leave=False):
        x = x.to(device, non_blocking=True)
        logits, _ = model(x)
        probs = torch.sigmoid(logits)
        all_labels.append(y.numpy())
        all_probs.append(probs.cpu().numpy())
    if not all_labels:
        return Metrics()
    labels = np.concatenate(all_labels)
    probs = np.concatenate(all_probs).flatten()
    preds = (probs >= threshold).astype(int)
    return compute_metrics(labels, preds, probs)


def get_cosine_schedule_with_warmup(optimizer, warmup_epochs, total_epochs, min_lr=1e-7):
    base_lr = optimizer.defaults['lr']
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return max(min_lr / base_lr, epoch / max(warmup_epochs, 1))
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return max(min_lr / base_lr, 0.5 * (1 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

---
## 7. Load Checkpoint

In [ ]:
CHECKPOINT_DIR = Path("/content/drive/MyDrive/DeepfakeDetectionModels")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_CKPT = CHECKPOINT_DIR / "gradient_field_cnn_v3.pth"
FINETUNE_CKPT = CHECKPOINT_DIR / cfg.checkpoint_name

# Load v3 model
model = CompactGradientNet(
    depth=cfg.depth, base_filters=cfg.base_filters,
    dropout=cfg.dropout, embedding_dim=cfg.embedding_dim
).to(device)

print(f"Loading checkpoint from {SOURCE_CKPT}")
checkpoint = torch.load(SOURCE_CKPT, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model"], strict=True)

epoch_loaded = checkpoint.get("epoch", "?")
auroc_loaded = checkpoint.get("best_auroc", "?")
print(f"✅ Loaded v3 checkpoint (epoch={epoch_loaded}, AUROC={auroc_loaded})")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Baseline evaluation (on validation set)
print("\n--- Baseline evaluation (before fine-tuning) ---")
baseline_metrics = evaluate(model, val_loader, device)
print(f"Baseline: Acc={baseline_metrics.accuracy:.4f} F1={baseline_metrics.f1:.4f} "
      f"AUROC={baseline_metrics.auroc:.4f} AP={baseline_metrics.avg_precision:.4f}")

---
## 8. Phase 1 — Head Tune (Frozen Backbone)

In [ ]:
print("\n" + "="*80)
print("PHASE 1: HEAD TUNE — Backbone frozen, training classifier head only")
print("="*80)

# Freeze everything except embedding + classifier
for name, param in model.named_parameters():
    if "embedding" in name or "classifier" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable: {trainable:,} | Frozen: {frozen:,}")

criterion = nn.BCEWithLogitsLoss()
head_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.head_lr, weight_decay=cfg.weight_decay
)

best_auroc = baseline_metrics.auroc if not np.isnan(baseline_metrics.auroc) else -1.0
head_history = {"train_loss": [], "val_metrics": []}

for epoch in range(cfg.head_epochs):
    t0 = time.time()
    tr_loss, tr_acc = one_epoch(model, train_loader, head_optimizer, criterion, device)
    val_m = evaluate(model, val_loader, device)
    dt = time.time() - t0

    head_history["train_loss"].append(tr_loss)
    head_history["val_metrics"].append(val_m)

    marker = ""
    if not np.isnan(val_m.auroc) and val_m.auroc > best_auroc:
        best_auroc = val_m.auroc
        torch.save({"model": model.state_dict(), "epoch": f"head_{epoch+1}",
                     "best_auroc": best_auroc, "phase": "head_tune"},
                    FINETUNE_CKPT)
        marker = " ✅ saved"

    print(f"[Head] Epoch {epoch+1}/{cfg.head_epochs} | {dt:.1f}s | "
          f"Loss={tr_loss:.4f} TrainAcc={tr_acc:.4f} | "
          f"ValAcc={val_m.accuracy:.4f} F1={val_m.f1:.4f} "
          f"AUROC={val_m.auroc:.4f} AP={val_m.avg_precision:.4f}{marker}")

print(f"\nPhase 1 complete. Best AUROC: {best_auroc:.4f}")

---
## 9. Phase 2 — Full Fine-Tune (All Layers Unfrozen)

In [ ]:
print("\n" + "="*80)
print("PHASE 2: FULL FINE-TUNE — All layers unfrozen")
print("="*80)

# Unfreeze all
for param in model.parameters():
    param.requires_grad = True

# Discriminative LR: backbone gets full_lr, head gets lower
backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if "embedding" in name or "classifier" in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

full_optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": cfg.full_lr},
    {"params": head_params, "lr": cfg.full_lr / 3},  # head at 1e-5
], weight_decay=cfg.weight_decay)

scheduler = get_cosine_schedule_with_warmup(
    full_optimizer, cfg.warmup_epochs, cfg.full_epochs, cfg.min_lr
)

patience_counter = 0
full_history = {"train_loss": [], "val_metrics": [], "lr": []}

for epoch in range(cfg.full_epochs):
    t0 = time.time()
    current_lr = full_optimizer.param_groups[0]["lr"]

    tr_loss, tr_acc = one_epoch(
        model, train_loader, full_optimizer, criterion, device,
        use_mixup=True, mixup_alpha=cfg.mixup_alpha, grad_clip=cfg.grad_clip
    )
    val_m = evaluate(model, val_loader, device)
    scheduler.step()
    dt = time.time() - t0

    full_history["train_loss"].append(tr_loss)
    full_history["val_metrics"].append(val_m)
    full_history["lr"].append(current_lr)

    marker = ""
    if not np.isnan(val_m.auroc) and val_m.auroc > best_auroc:
        best_auroc = val_m.auroc
        patience_counter = 0
        torch.save({"model": model.state_dict(), "epoch": f"full_{epoch+1}",
                     "best_auroc": best_auroc, "phase": "full_finetune"},
                    FINETUNE_CKPT)
        marker = " ✅ saved"
    else:
        patience_counter += 1

    print(f"[Full] Epoch {epoch+1}/{cfg.full_epochs} | {dt:.1f}s | LR={current_lr:.2e} | "
          f"Loss={tr_loss:.4f} TrainAcc={tr_acc:.4f} | "
          f"ValAcc={val_m.accuracy:.4f} F1={val_m.f1:.4f} "
          f"AUROC={val_m.auroc:.4f} AP={val_m.avg_precision:.4f}{marker}")

    if patience_counter >= cfg.patience:
        print(f"  ⏹️ Early stopping at epoch {epoch+1}")
        break

# Load best checkpoint
ckpt = torch.load(FINETUNE_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
print(f"\n✅ Fine-tuning complete. Best AUROC: {best_auroc:.4f}")
print(f"   Checkpoint: {FINETUNE_CKPT}")

---
## 10. Evaluation & Comparison

In [ ]:
finetuned_metrics = evaluate(model, test_loader, device)

print(f"\n{'='*60}")
print(f"{'BASELINE vs FINE-TUNED COMPARISON':^60}")
print(f"{'='*60}")
print(f"{'Metric':<20} {'Baseline':>12} {'Fine-tuned':>12} {'Delta':>10}")
print(f"{'-'*60}")
for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'specificity', 'auroc', 'avg_precision']:
    b = getattr(baseline_metrics, metric_name)
    f = getattr(finetuned_metrics, metric_name)
    delta = f - b
    sign = "+" if delta >= 0 else ""
    print(f"{metric_name:<20} {b:>12.4f} {f:>12.4f} {sign}{delta:>9.4f}")
print(f"{'='*60}")

# ROC overlay
fig, ax = plt.subplots(figsize=(7, 6))
for label, m, color in [("Baseline v3", baseline_metrics, "#888"), ("Fine-tuned", finetuned_metrics, "#2196F3")]:
    if not np.isnan(m.auroc):
        fpr, tpr, _ = roc_curve(m.labels, m.probs)
        ax.plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUROC={m.auroc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC — Baseline vs Fine-tuned")
ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "gradfield_finetune_roc.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 11. Export for Fusion

In [ ]:
EXPORT_DIR = Path("../models/grad_field_cnn")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

torch.save({"model_state": model.state_dict()}, EXPORT_DIR / "weights_finetuned.pt")
torch.save(model.state_dict(), EXPORT_DIR / "model_finetuned.pth")

config = {
    "name": "gradfield-cnn", "framework": "pytorch",
    "arch": "CompactGradientNet", "version": "v3-finetuned",
    "model_parameters": {"depth": cfg.depth, "base_filters": cfg.base_filters,
                         "dropout": cfg.dropout, "embedding_dim": cfg.embedding_dim},
    "finetune_config": asdict(cfg),
    "best_auroc": float(best_auroc),
}
with open(EXPORT_DIR / "config_finetuned.json", "w") as f:
    json.dump(config, f, indent=2)

# ROC data for paper overlay
fpr_ft, tpr_ft, _ = roc_curve(finetuned_metrics.labels, finetuned_metrics.probs)
roc_data = {
    "branch": "Gradient Field CNN (fine-tuned)", "auroc": round(float(finetuned_metrics.auroc), 4),
    "fpr": [round(float(x), 6) for x in fpr_ft],
    "tpr": [round(float(x), 6) for x in tpr_ft],
}
with open(EXPORT_DIR / "gradfield_roc_data_finetuned.json", "w") as f:
    json.dump(roc_data, f, indent=2)

print(f"✅ Exported fine-tuned model to {EXPORT_DIR}")

---
## 12. Per-Split Evaluation

In [ ]:
print("\n" + "="*60)
print("PER-SPLIT EVALUATION")
print("="*60)

for split_name, loader in [("Validation", val_loader), ("Test", test_loader)]:
    m = evaluate(model, loader, device)
    print(f"\n{split_name}: Acc={m.accuracy:.4f} F1={m.f1:.4f} "
          f"AUROC={m.auroc:.4f} AP={m.avg_precision:.4f}")

print(f"\n✅ Evaluation complete")